# Pandas Cleaning Case Study

**Decision problem:** what cleaning decisions are required before a dataset can support a recommendation?

**Goal:** build a clean, documented analytical table from a raw CSV snapshot.

## Cleaning checklist

- Load data.
- Standardize column names.
- Parse dates.
- Convert numeric columns.
- Handle missing values.
- Validate output shape and ranges.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd()
LOCAL_DATA = ROOT / "data" / "snapshots"
if not LOCAL_DATA.exists():
    LOCAL_DATA = ROOT.parent / "data" / "snapshots"
print("Data path:", LOCAL_DATA.resolve())

In [ ]:
path = sorted(LOCAL_DATA.glob("*.csv"))[0]
raw = pd.read_csv(path)
raw.head()

In [ ]:
clean = raw.copy()
clean.columns = (clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace(":", "_", regex=False))
clean = clean.rename(columns={clean.columns[0]: "date"})
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
clean.head()

In [ ]:
signal_cols = [c for c in clean.columns if c not in {"date", "ispartial", "is_partial"}]
for col in signal_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

clean = clean.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
clean[signal_cols] = clean[signal_cols].ffill().bfill()
clean.head()

In [ ]:
quality_report = pd.DataFrame({
    "dtype": clean.dtypes.astype(str),
    "missing": clean.isna().sum(),
    "unique": clean.nunique()
})
quality_report

In [ ]:
assert clean["date"].is_monotonic_increasing
assert clean[signal_cols].notna().all().all()
assert (clean[signal_cols].min().min() >= 0)
assert (clean[signal_cols].max().max() <= 100)
print("Cleaning checks passed:", clean.shape)

In [ ]:
clean_long = clean.melt(id_vars="date", value_vars=signal_cols, var_name="signal", value_name="index")
clean_long.head()

## Deliverable

Export or publish: cleaned table, quality report, and 5-line data dictionary.